[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/04_evaluation.ipynb)

# 04 — Evaluation and Method Comparison

**Inputs.** The newest finished run of each method and seed under `outputs/optim/`, the baseline radio map, and the
processed tables. **Outputs.** `reports/tables/04_evaluation/` and `reports/figures/04_evaluation/`.

No GPU is needed and nothing is re-traced. Every table and figure is built by `src/evaluation/run.py`, the same code
`task evaluate` runs; this notebook only presents them.

In [1]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from src.config import load_config
from src.evaluation.run import evaluate

# evaluate() seeds, sets the figure defaults and saves every table and figure itself.
cfg = load_config(overrides=CONFIG_OVERRIDES)
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

Build every table and figure once. Run directories that did not finish writing are skipped with a `WARNING` line.

In [3]:
results = evaluate(cfg, in_colab=IN_COLAB)

## 1. Evaluation Objective

This notebook compares search methods for **multi-band antenna tilt coordination**: one absolute tilt per cell-band
pair, chosen to

* reduce coverage holes,
* reduce co-band coverage overlap and the number of overlapping neighbours,
* reduce weak coverage,
* raise the signal and interference levels where coverage exists,
* serve more UEs within the cells' PRB capacity,
* keep the load off any one cell and spread evenly across them,

at a reasonable number of expensive ray-tracing evaluations. The eleven KPIs are reported side by side and weighed
equally; none is ranked above another, and each is reported over all bands and per band. The single number a search
maximises is the objective $J$ of `docs/adr/0007-demand-weighted-objective.md`, which is reported beside the KPIs, not
in place of them. Unlike the KPIs, $J$ is **demand-weighted**: it averages a per-tile coverage utility against the
demand map notebook 02 builds from the MDT, so the two disagree on purpose — a KPI says how much of the *map* is bad,
$J$ says how much of the *traffic* sits where it is bad.

## 2. Methods Under Evaluation

| Method | Key | Type | Description |
|---|---|---|---|
| Current configuration | `incumbent` | Reference | The committed tilts, evaluation 0 of every run |
| Random search | `random` | Baseline | Scrambled Sobol samples over the tilt box; its first `n_init` points are TuRBO's initial design (`configs/optim/method/random.yaml`) |
| Rule-based sweep | `rule` | Baseline | Operator heuristic: one shared tilt per band, coordinate descent over a grid of values (`configs/optim/method/rule.yaml`) |
| TuRBO | `turbo` | Proposed | TuRBO-1 trust-region Bayesian optimisation on $J$ (`configs/optim/method/turbo.yaml`, ADR 0003/0006) |

Random search and TuRBO share seed and evaluation budget, so they differ only in where they look. The rule-based
sweep is not budget-matched. There is no separate plain-random arm apart from Sobol, and the Multi-Agent
Reinforcement Learning arm is not implemented.

## 3. Experimental Setup

### 3.1 Tilt search space

For each cell-band pair $(i,b)$ the decision variable is the absolute tilt $\theta_{i,b}$, continuous within its bounds:

$$
\theta^{\min}_{i,b} \le \theta_{i,b} \le \theta^{\max}_{i,b}.
$$

No tilt step and no maximum change from the current tilt $\theta^{(0)}$ are imposed; how far antennas moved is
reported in Section 13, not constrained.

### 3.2 Network, search space and budget

UE counts are over every UE, in the search and here alike (`src/evaluation/__init__.py`).

In [4]:
display(results["experiment_setup"])

,Parameter,Setting
0,Scenario,scn_7d938e15f9ac4618
1,Cells,12
2,Frequency bands,"2600 MHz, 1800 MHz, 700 MHz"
3,Decision variables (cell-band tilts),36
4,Evaluation area [m],6200 x 6520
5,Grid resolution [m],20
6,Grid tiles,101060
7,UE reports,10087
8,Measurement intervals,672
9,Tilt bounds [°],0 to 15


## 4. Evaluation KPIs

Notation: $R_{i,b}(g)$ is the RSRP of cell $i$ on band $b$ at tile $g \in G$, and $R_s(g) = \max_{(i,b)} R_{i,b}(g)$ the
strongest layer. The thresholds $T_{\text{hole}}$, $T_{\text{weak}}$ and $\Delta_{\text{overlap}}$ are
`kpi.hole_dbm`, `kpi.weak_dbm` and `kpi.overlap_margin_db` in `configs/kpi.yaml`; their values are in the setup table
above. The implementations are in `src/kpi/`, and every one of the eleven is also reported **per frequency layer** by
giving the same function one band's slice of the radio map (section 11).

### 4.1 Coverage hole rate (minimise)

$$
\mathrm{HoleRate} = \frac{1}{|G|}\sum_{g\in G}\mathbb{1}\left[R_s(g)\le T_{\text{hole}}\right]
$$

A tile with no ray-traced path counts as a hole.

### 4.2 Co-band overlap rate (minimise)

Overlap is **co-band**: within band $b$, the strongest transmitter serves, and each other transmitter $j$ on that band
is an overlapping neighbour when

$$
R_{j,b}(g) > T_{\text{hole}} \quad\text{and}\quad R_{j,b}(g) \ge \max_i R_{i,b}(g) - \Delta_{\text{overlap}}.
$$

$N_{\text{ov}}(g)$ is that count summed over bands, and $\mathrm{OverlapRate} = |\{g : N_{\text{ov}}(g) > 0\}| / |G|$.

### 4.3 Overlap neighbours per covered tile (minimise)

$$
\mathrm{OverlapNeighborMean} = \frac{1}{|G_{\text{covered}}|}\sum_{g\in G_{\text{covered}}} N_{\text{ov}}(g),
\qquad G_{\text{covered}} = \{g : R_s(g) > T_{\text{hole}}\}.
$$

The severity behind 4.2's incidence: it separates one competing neighbour from a pile-up of many.

### 4.4 Weak coverage rate (minimise)

$$
\mathrm{WeakRate} = \frac{1}{|G|}\sum_{g\in G}\mathbb{1}\left[T_{\text{hole}} < R_s(g) \le T_{\text{weak}}\right]
$$

### 4.5 RSRP percentiles, $\mathrm{RSRP}_{p05}$ and $\mathrm{RSRP}_{p50}$ (maximise)

The 5th percentile and the median of $R_s(g)$ over $G_{\text{covered}}$. The 5% point is the cell-edge measure of
3GPP TR 36.814 Annex A.2.1.4. Both are **conditional on coverage**, so a configuration can raise them by covering
less: read them beside the hole rate, never alone.

### 4.6 SINR percentiles, $\mathrm{SINR}_{p05}$ and $\mathrm{SINR}_{p50}$ (maximise)

The same two points of the solver's SINR at the layer $R_s(g)$ names, over the same tiles, so 4.5 and 4.6 describe the
same cell-band at every tile. The SINR is the solver's own: it assumes every co-band transmitter is fully loaded, and
is not recomputed against the PRB load the serving rule produces.

### 4.7 Served UE rate (maximise)

The share of UE reports admitted to a cell-band above $T_{\text{hole}}$ under the serving rule and the PRB limits of
`src/kpi/capacity.py`. The capacity settings are placeholders, so read its direction rather than its level. A UE on a
hole and a UE blocked everywhere both count as not served.

### 4.8 Peak PRB utilisation (minimise)

$$
\mathrm{PrbUtilisationMax} = \max_{t}\max_{(i,b)} \frac{\mathrm{PRB}_{i,b}(t)}{\mathrm{maxPRB}_{i,b}}
$$

over every interval $t$ and cell-band. The serving rule refuses any admission that would carry a cell-band past
`kpi.capacity.max_admission_utilisation`, so this measure is bounded by that value by construction; a run at the
ceiling is a network turning traffic away, and the headroom below it is what absorbs a tilt change.

### 4.9 Cell load imbalance (minimise)

The coefficient of variation of cell-band utilisation: each cell-band's utilisation is averaged over every interval
(idle ones as zero), and the population standard deviation of those averages is divided by their mean. Scale-free — it
does not move when the whole network gets busier, only when the traffic sits unevenly, which is what a tilt change can
fix. Zero means every cell-band carries the same share of its limit.

### 4.10 Objective $J$ (maximise)

$$
J = \frac{\sum_{g\in G} w_g\, \lambda_g\, e^{1 - \lambda_g}}{\sum_{g\in G} w_g},
\qquad \lambda_g = 1 + N_{\text{ov},b(g)}(g),
\qquad w_g = 1 + r_g
$$

$b(g)$ is the first band of `kpi.capacity.band_preference` whose strongest cell clears $T_{\text{hole}}$ — the band
the serving rule of 4.7 would admit a UE on — and $N_{\text{ov},b(g)}$ is the 4.2 neighbour count **on that band
alone**, not the band-summed $N_{\text{ov}}$. So $\lambda_g$ counts the cells contending to serve the tile, and
$\lambda_g = 0$ where no band covers it.

$\lambda e^{1-\lambda}$ is worth exactly 1 at $\lambda = 1$, 0.736 at 2, 0.406 at 3 and 0 at 0, so the objective pays
for one dominant server and nothing else; a tile at $\lambda = 1$ is the one the ADR calls *effectively covered*.
$r_g$ is the tile's MDT reports over the busiest tile's (notebook 02, section 7), so $w_g \in [1, 2]$: demand doubles
a tile at most and silences none.

$J \in [0, 1]$ and is **maximised**; it reaches 1 only if every tile of the grid has exactly one serving cell. It has
no free parameters — it reads `kpi.hole_dbm`, `kpi.overlap_margin_db` and `kpi.capacity.band_preference`, all of which
the KPIs above already use. See [ADR 0009](../docs/adr/0009-effective-coverage-objective.md).


## 5. Result Loading and Validation

`evaluate` loaded the newest finished run of each method and seed; unfinished run directories were skipped above. Runs are compared only if they share the scenario, grid,
solver settings, bands and KPI definition; otherwise `evaluate` raises. The recorded KPIs are then recomputed from
each archived radio map: any visible gap means the archived map is not the map that was scored.

In [5]:
checks = results["comparability_checks"]
print(f"{int(checks['Holds'].sum())} of {len(checks)} comparability checks hold")
display(checks)
reproducibility = results["kpi_reproducibility"]
print(f"Largest recorded-vs-recomputed gap: {reproducibility['Absolute gap'].max():.2e}")

23 of 23 comparability checks hold


,Check,Holds,Offending runs
0,every run optimized the baseline's scenario,True,
1,grid n_rows matches the baseline,True,
2,grid n_cols matches the baseline,True,
3,grid tile_size_m matches the baseline,True,
4,grid origin_x matches the baseline,True,
5,grid origin_y matches the baseline,True,
6,grid ue_height_m matches the baseline,True,
7,solver samples_per_tx matches the baseline,True,
8,solver max_depth matches the baseline,True,
9,solver los matches the baseline,True,


Largest recorded-vs-recomputed gap: 2.26e-06


**Observations.** All 23 comparability checks hold, and every run records the same KPI definition, capacity model and objective parameters - including the per-band `alpha`, which `runs.verify` now guards in place of the demand map's former KDE settings (the map has none left). The KPIs recomputed from each archived radio map match the recorded ones to at most 1.9e-06 (the cell-edge RSRP percentile); the objective itself matches to under 1e-09. The winner's map is re-traced after the search and GPU ray tracing is not bit-reproducible, so that is the size of the solver's own noise, not a bookkeeping error. The three runs measure the same thing, so their differences come from tilt alone. One search seed (42) per method was found.

## 6. Final KPI Comparison

### 6.1 Best configuration per method

Each method's winner, averaged over seeds, against the current configuration. With one seed per method the spread and
interval columns are empty.

In [6]:
display(results["kpi_scoreboard"])
display(results["method_cost"])

,Method,KPI,Direction,Seeds,Current configuration,Mean,Standard deviation,95% CI lower,95% CI upper,Mean change,Verdict
0,Random search,Coverage hole rate,minimise,1,0.1125,0.1094,NaN,NaN,NaN,-3.0873e-03,better
1,Random search,Co-band overlap rate,minimise,1,0.3164,0.3368,NaN,NaN,NaN,2.0414e-02,worse
2,Random search,Overlap neighbours per covered tile,minimise,1,0.9625,1.0501,NaN,NaN,NaN,8.7570e-02,worse
3,Random search,Weak coverage rate,minimise,1,0.3069,0.3063,NaN,NaN,NaN,-6.6297e-04,better
4,Random search,"Cell-edge RSRP, p05 [dBm]",maximise,1,-108.5644,-108.4613,NaN,NaN,NaN,1.0307e-01,better
5,Random search,"Median RSRP, p50 [dBm]",maximise,1,-84.1068,-83.8138,NaN,NaN,NaN,2.9301e-01,better
6,Random search,"Cell-edge SINR, p05 [dB]",maximise,1,-5.8729,-5.8538,NaN,NaN,NaN,1.9010e-02,better
7,Random search,"Median SINR, p50 [dB]",maximise,1,8.4932,8.3016,NaN,NaN,NaN,-1.9162e-01,worse
8,Random search,Served UE rate,maximise,1,0.5265,0.5758,NaN,NaN,NaN,4.9271e-02,better
9,Random search,Peak PRB utilisation,minimise,1,0.7999,0.7999,NaN,NaN,NaN,-5.6731e-05,better


,Method,Seed,Run,Evaluations,Best evaluation,Ray tracing [min],Wall clock [min],Coverage hole rate,Co-band overlap rate,Overlap neighbours per covered tile,Weak coverage rate,"Cell-edge RSRP, p05 [dBm]","Median RSRP, p50 [dBm]","Cell-edge SINR, p05 [dB]","Median SINR, p50 [dB]",Served UE rate,Peak PRB utilisation,Cell load imbalance (CoV),Objective J,KPIs improved,KPIs worsened
0,Random search,42,2026-09-20_04-09-52,145,62,7.4861,10.2194,0.1094,0.3368,1.0501,0.3063,-108.4613,-83.8138,-5.8538,8.3016,0.5758,0.7999,1.0805,0.8155,8,4
1,Rule-based sweep,42,2026-09-20_04-20-14,28,11,1.0731,1.8539,0.1044,0.3247,1.1203,0.2538,-106.9844,-80.8430,-5.0047,9.8325,0.5930,0.7992,0.9416,0.8189,9,3
2,TuRBO,42,2026-09-20_03-50-06,145,144,9.0052,19.5926,0.1058,0.3402,1.0643,0.2638,-107.4161,-81.4024,-5.1535,9.6435,0.5787,0.7997,1.0325,0.8231,9,3


### 6.2 Improvement over the current configuration

$$
\Delta K = \pm\,\frac{K_{\text{optimised}} - K_{\text{initial}}}{|K_{\text{initial}}|}\times 100\,\%,
$$

signed so that a positive value is an improvement whichever direction the KPI runs.

In [7]:
display(results["kpi_relative_improvement"])
display(results["kpi_improvement"])

,Method,Coverage hole rate,Co-band overlap rate,Overlap neighbours per covered tile,Weak coverage rate,"Cell-edge RSRP, p05 [dBm]","Median RSRP, p50 [dBm]","Cell-edge SINR, p05 [dB]","Median SINR, p50 [dB]",Served UE rate,Peak PRB utilisation,Cell load imbalance (CoV),Objective J
0,Random search,2.7445,-6.4519,-9.0979,0.2160,0.0949,0.3484,0.3237,-2.2562,9.3579,0.0071,-18.7682,0.9222
1,Rule-based sweep,7.1605,-2.6114,-16.3892,17.2937,1.4554,3.8806,14.7832,15.7691,12.6342,0.0884,-3.4926,1.3452
2,TuRBO,5.9553,-7.5309,-10.5725,14.0407,1.0577,3.2154,12.2481,13.5432,9.9040,0.0227,-13.4922,1.8631


<Figure size 1540x1155 with 12 Axes>

**Observations.** TuRBO raises the objective most: **+1.86 % relative, 0.8080 to 0.8231**. The rule-based sweep gains +1.35 % and random search +0.92 %. On KPI counts the sweep and TuRBO tie at 9 better and 3 worse; random search is 8 and 4.

$J$ is the demand-weighted share of the grid served cleanly by exactly one cell (ADR 0009), bounded in $[0, 1]$, with no free parameters. None of these values compares with anything recorded before that change. The headroom is small by construction: from an incumbent of 0.8080 the most any configuration could gain is 0.1920, so TuRBO's +0.0151 is 7.9 % of everything available, the sweep's +0.0109 is 5.7 % and random search's +0.0075 is 3.9 %.

**The objective and the KPI set disagree about the winner, and that is the finding here.** TuRBO wins J and wins the quantity J is built from - effective coverage, the share of tiles with exactly one serving cell, rises from 66.1 % to 69.6 %, the best of the three. But **the rule sweep beats TuRBO on 8 of the 11 KPIs**: hole rate (0.1044 against 0.1058), weak rate (0.2538 against 0.2638), both RSRP percentiles, both SINR percentiles, served rate (0.5930 against 0.5787) and peak PRB utilisation. The reason is structural: J does not read signal strength above the hole threshold, and that is where most of the sweep's advantage lies. Median RSRP rises 3.3 dB under the sweep against 2.7 dB under TuRBO, and none of it scores.

Three measures move the wrong way under every method. **The band-collapsed co-band overlap rate** rises (0.3164 to 0.3247, 0.3368 and 0.3402) and **overlap neighbours per covered tile** rises with it (+16.4 %, +9.1 %, +10.6 %). Both follow from the objective's exchange rate: lifting a tile out of a hole gains the full 1.000 of utility, splitting a clean tile in two costs only $1 - 2e^{-1} = 0.264$, so a search will accept 3.78 newly crowded tiles per hole closed. Filling holes also brings new tiles into the covered set, and a tile at the edge of two footprints arrives with a neighbour already. Section 11 shows that on the one band the objective reads, overlap *improves* under all three. **Cell load imbalance** also worsens everywhere (+3.5 %, +18.8 %, +13.5 %); nothing in J asks for even load.

## 7. Optimization Convergence

Best objective found so far against the number of evaluations, the current configuration being evaluation 0. The
band is the min–max over seeds and collapses to the line with one seed.

The second table asks whether the search mattered: each winner against the typical candidate its own search evaluated.

In [8]:
display(results["search_progress"])
display(results["winner_vs_candidates"])

<Figure size 990x550 with 1 Axes>

,Method,Seed,Current configuration,"Initial design, median objective","All candidates, median objective","All candidates, 90th percentile objective",Best objective found
0,Random search,42,0.808,0.8017,0.8019,0.8082,0.8155
1,Rule-based sweep,42,0.808,NaN,0.8128,0.8184,0.8189
2,TuRBO,42,0.808,0.8017,0.8180,0.8217,0.8231


**Observations.** TuRBO's median candidate (0.8180) already scores above random search's best (0.8155), and its 90th percentile (0.8217) above the rule sweep's best (0.8189). The trust region spends its budget near good configurations. Random search's candidates sit around their Sobol design median (0.8017) - *below* the current configuration's 0.8080, so the committed per-band tilts are a reasonable starting point rather than a weak one - and it found its best at evaluation 62. The rule sweep's median candidate (0.8128) is high because it only moves whole bands around the current tilts, so it never visits a bad region.

Splitting TuRBO's own evaluations by what proposed them makes the case directly: the 16 shared Sobol points averaged 0.8019, and TuRBO's 128 trust-region proposals averaged **0.8176**. The two runs share those 16 points and diverge only once the model starts proposing.

## 8. Sample Efficiency

Best value reached after a fixed number of evaluations, averaged over seeds. Each KPI is its own running best, so the
rows of one budget need not come from one configuration. The last budget is the longest run; a budget past a run's
length is empty.

In [9]:
display(results["sample_efficiency"])

,KPI,Evaluations,Random search,Rule-based sweep,TuRBO
0,Objective J,10,0.8080,0.8173,0.8080
1,Objective J,25,0.8127,0.8189,0.8126
2,Objective J,50,0.8127,NaN,0.8186
3,Objective J,100,0.8155,NaN,0.8211
4,Objective J,145,0.8155,NaN,0.8231
5,Coverage hole rate,10,0.1083,0.1079,0.1083
6,Coverage hole rate,25,0.1083,0.1042,0.1083
7,Coverage hole rate,50,0.1082,NaN,0.1073
8,Coverage hole rate,100,0.1082,NaN,0.1063
9,Coverage hole rate,145,0.1082,NaN,0.1054


**Observations.** The rule-based sweep reaches J = 0.8189 after 25 evaluations and stops at 28. TuRBO is still at 0.8186 after 50, reaches 0.8211 by 100 and ends at 0.8231 after 145 - so **the sweep is the better answer on any budget up to about 50 evaluations**, and TuRBO the better one above it. Random search reaches 0.8127 by evaluation 25 and only improves once more, to 0.8155 at evaluation 62.

The running-best overlap rate tells a blunter story: **no candidate of any method ever beat the incumbent's 0.3164 except the rule sweep's**, which found 0.3067 within its first 10 evaluations and then discarded it, because J scored another candidate higher. Overlap-reducing configurations were available and the objective did not select them.

## 9. Multi-Objective Trade-Offs and Pareto Fronts

Each point is one evaluated configuration. The stars mark each method's highest-$J$ configuration, the
cross marks the current configuration, and the dashed line joins the Pareto-efficient configurations over all methods:
those that no other configuration beats on both axes at once.

The served UE ratio takes the place of the template's band-priority axis, which is not used in this project.

In [10]:
display(results["tradeoff_hole_rate_vs_overlap_rate"])
display(results["tradeoff_hole_rate_vs_served_rate"])
display(results["tradeoff_overlap_rate_vs_served_rate"])

<Figure size 880x605 with 1 Axes>

<Figure size 880x605 with 1 Axes>

<Figure size 880x605 with 1 Axes>

**Observations.** Hole rate and overlap rate conflict along the Pareto front: every step towards fewer holes costs overlap, at the 3.78-to-1 exchange rate the objective sets. All three picks sit at the low-hole end and above the current configuration's overlap rate of 0.3164 - the sweep at 0.3247, random search at 0.3368 and TuRBO at 0.3402. The front's low-overlap end is populated but unvisited by any winner, which is the clearest single picture of what this objective does and does not price.

Against the served rate the picture is different: all three picks sit close together near 0.58-0.59, because the served rate is bounded by the 0.8 admission ceiling rather than by coverage, so it cannot be traded far in either direction.

## 10. Spatial Coverage Analysis

### 11.1 Change in best-server RSRP per method

In [11]:
display(results["rsrp_change_maps"])

<Figure size 1562x484 with 4 Axes>

### 10.2 Before and after: the recommended configuration

The recommended configuration is the run with the highest objective over all methods. Both panels use the same scale
and the same coverage classes.

In [12]:
display(results["coverage_before_after"])
display(results["coverage_class_maps"])
display(results["coverage_by_area_and_demand"])

<Figure size 1650x506 with 6 Axes>

<Figure size 1166x506 with 3 Axes>

,Coverage class,Current configuration: Share of area,Current configuration: Share of demand,Random search: Share of area,Random search: Share of demand,Rule-based sweep: Share of area,Rule-based sweep: Share of demand,TuRBO: Share of area,TuRBO: Share of demand
0,hole,0.1125,0.000,0.1094,0.0037,0.1044,0.00,0.1058,0.0000
1,weak,0.3069,0.836,0.3063,0.8037,0.2538,0.77,0.2638,0.7777
2,good,0.5806,0.164,0.5843,0.1925,0.6417,0.23,0.6304,0.2223


### 10.3 Overlap neighbours

In [13]:
display(results["overlap_neighbour_summary"])
display(results["overlap_neighbour_maps"])

,Configuration,"Mean overlap neighbours, covered tiles","Mean overlap neighbours, all tiles",Share with 0 neighbours,Share with 1 neighbour,Share with 2 neighbours,Share with 3+ neighbours
0,Current configuration,0.9625,0.8543,0.6435,0.1127,0.0677,0.1761
1,Random search,1.0501,0.9352,0.6218,0.1339,0.0789,0.1654
2,Rule-based sweep,1.1203,1.0033,0.6375,0.1082,0.0757,0.1787
3,TuRBO,1.0643,0.9517,0.6195,0.1374,0.0827,0.1603


<Figure size 1078x484 with 3 Axes>

**Observations.** The recommended TuRBO configuration improves coverage by area on every class: holes 11.2 % to 10.6 %, weak 30.7 % to 26.4 %, good 58.1 % to 63.0 %. Weighted by where the served traffic stands, no demand ends up on hole tiles (none before either) while the share on weak tiles falls from 83.6 % to 77.8 % and the share on good tiles rises from 16.4 % to 22.2 %.

Overlap neighbours move two ways at once, and it is worth reading both columns. The share of covered tiles with **three or more** neighbours falls from 17.6 % to 16.0 %, the best of the three methods - the worst pile-ups thin out. But the share with **no** neighbour also falls, 64.4 % to 62.0 %, and the *mean* over covered tiles rises from 0.963 to 1.064, because the tiles TuRBO newly covers arrive at the edge of two footprints and carry a neighbour with them.

That is exactly the shape of $\lambda e^{1-\lambda}$: the penalty is steepest between one and three contending cells and nearly flat beyond four, so the objective will happily convert a few badly contested tiles into many mildly contested ones. The rule sweep, which has no per-cell freedom to do this, raises the mean further (to 1.120) and thickens the pile-ups instead (17.6 % to 17.9 %).

## 11. Frequency-Layer Analysis

Per band: the area the band covers above the hole threshold and its mean RSRP there, the area the serving rule assigns
to it, and the share of UE reports actually admitted on it after PRB limits, with their median SINR. The band preference
of the serving rule is `kpi.capacity.band_preference`.

Then every KPI of section 4 again, per layer. The `all` row is the whole radio map and equals the scoreboard above; a
band row is the same function given that band's layers, so nothing is redefined. The per-band served rates are shares
of *all* reports and so sum to the `all` row; the rates over tiles do not sum to anything, because a tile can be a hole
on two bands at once. There is no per-band objective: $J$ scores a network, and one layer of a multi-band network is
not a network.

In [14]:
display(results["band_layer_summary"])
display(results["serving_band_mix"])
display(results["ue_service_summary"])
display(results["band_kpis"])
display(results["band_kpi_panels"])

,Configuration,Band,Share of area covered by the band,Mean band RSRP where covered [dBm],Share of area served on the band,Share of UE reports served on the band,"Served SINR on the band, median [dB]"
0,Current configuration,2600 MHz,0.7116,-97.1547,0.7116,0.3408,3.4679
1,Current configuration,1800 MHz,0.7650,-91.4508,0.0584,0.1050,5.1757
2,Current configuration,700 MHz,0.8619,-84.2119,0.1176,0.0807,13.3089
3,Random search,2600 MHz,0.7369,-92.0374,0.7369,0.4113,5.6267
4,Random search,1800 MHz,0.7741,-88.9841,0.0443,0.0953,6.5005
5,Random search,700 MHz,0.8627,-84.1921,0.1094,0.0692,13.6932
6,Rule-based sweep,2600 MHz,0.7382,-92.5080,0.7382,0.4379,6.2930
7,Rule-based sweep,1800 MHz,0.7830,-87.3601,0.0511,0.0816,5.3374
8,Rule-based sweep,700 MHz,0.8677,-81.8644,0.1063,0.0736,13.2039
9,TuRBO,2600 MHz,0.7235,-92.8946,0.7235,0.3310,6.1002


<Figure size 1122x495 with 1 Axes>

,Configuration,UE reports,Share not served,"Served SINR, 10th percentile [dB]","Served SINR, median [dB]","PRBs per served UE, median",Share served on 2600 MHz,Share served on 1800 MHz,Share served on 700 MHz
0,Current configuration,10087.0,0.4735,-0.3751,5.1257,53.1831,0.3408,0.1050,0.0807
1,Random search,10087.0,0.4242,0.1872,6.8475,43.6465,0.4113,0.0953,0.0692
2,Rule-based sweep,10087.0,0.4070,0.2463,6.9498,43.1678,0.4379,0.0816,0.0736
3,TuRBO,10087.0,0.4213,-0.1048,8.6232,36.4279,0.3310,0.1721,0.0755


,Configuration,Band,Coverage hole rate,Co-band overlap rate,Overlap neighbours per covered tile,Weak coverage rate,"Cell-edge RSRP, p05 [dBm]","Median RSRP, p50 [dBm]","Cell-edge SINR, p05 [dB]","Median SINR, p50 [dB]",Served UE rate,Peak PRB utilisation,Cell load imbalance (CoV)
0,Current configuration,All bands,0.1125,0.3164,0.9625,0.3069,-108.5644,-84.1068,-5.8729,8.4932,0.5265,0.7999,0.9098
1,Current configuration,2600 MHz,0.2884,0.2010,0.3703,0.5124,-115.4094,-98.0890,-17.5911,-1.6299,0.3408,0.7999,0.6775
2,Current configuration,1800 MHz,0.2350,0.2067,0.3564,0.4171,-112.4538,-91.7208,-11.6703,4.1494,0.1050,0.7994,0.6074
3,Current configuration,700 MHz,0.1381,0.2396,0.3691,0.2945,-108.5333,-83.9182,-5.2264,8.3972,0.0807,0.7991,0.9986
4,Random search,All bands,0.1094,0.3368,1.0501,0.3063,-108.4613,-83.8138,-5.8538,8.3016,0.5758,0.7999,1.0805
5,Random search,2600 MHz,0.2631,0.1788,0.3366,0.4147,-113.9745,-92.3950,-16.1740,2.6907,0.4113,0.7995,0.7194
6,Random search,1800 MHz,0.2259,0.2064,0.3866,0.3644,-112.0356,-88.8772,-11.2691,5.5166,0.0953,0.7999,1.0369
7,Random search,700 MHz,0.1373,0.2496,0.4496,0.3002,-108.7323,-84.1073,-5.4767,7.7593,0.0692,0.7998,1.1554
8,Rule-based sweep,All bands,0.1044,0.3247,1.1203,0.2538,-106.9844,-80.8430,-5.0047,9.8325,0.5930,0.7992,0.9416
9,Rule-based sweep,2600 MHz,0.2618,0.1901,0.3385,0.4211,-114.0820,-92.7345,-16.2745,2.1617,0.4379,0.7992,0.5741


<Figure size 1518x748 with 6 Axes>

**Observations.** The per-band rows are the point of a multi-band study, and they say something the whole-network row cannot: **no single layer covers the area**. Alone, 2600 MHz leaves 28.8 % of tiles as holes, 1800 MHz 23.5 % and 700 MHz 13.8 %, against 11.2 % for the union. The layers are complementary, not redundant, and that is the premise the project rests on.

**But the objective only reads one of them per tile.** Under ADR 0009 each tile is scored on the most preferred band that clears -120 dBm - 2600 MHz on 71 % of tiles, 1800 MHz on 5.8 % and 700 MHz on 11.8 % - so the other layers are unpriced wherever a preferred one is available. The per-band overlap rows show the consequence directly: **2600 MHz overlap falls under every method** (0.2010 to 0.1718 under TuRBO, 0.1901 under the sweep, 0.1788 under random search), while the unscored 1800 MHz (to 0.2082) and 700 MHz (to 0.2440) layers drift slightly worse. The band-collapsed rate in the scoreboard rises because it is the union of all three plus the newly covered ground.

Per-band hole rates improve everywhere, most on the mid band: 1800 MHz 0.235 to 0.220 under TuRBO (-6.4 % relative), 2600 MHz 0.288 to 0.276 (-4.1 %) and 700 MHz 0.138 to 0.133 (-3.4 %).

**The serving mix is where TuRBO parts company with the baselines.** Both baselines push UE reports onto the preferred 2600 MHz layer (34.1 % to 43.8 % and 41.1 %). TuRBO drops it to 33.1 % and lifts **1800 MHz from 10.5 % to 17.2 %** instead, by shrinking a few 2600 MHz footprints toward the 15° bound and opening 1800 MHz almost to 0°. The payoff is the best service quality of the three: median PRBs per served UE fall from 53.2 to 36.4 and median served SINR rises from 5.1 to 8.6 dB, because traffic moved off the most contended layer.

Every band in every configuration peaks at about 0.799 of its PRB limit, the admission ceiling, so no layer has headroom.

## 12. Cell Load and PRB Usage

The two load KPIs are reductions of one series: the PRBs each cell-band carried in each 15-minute interval, as a share
of its own limit. The heatmap is that series; the table beside it is the per-cell-band summary the utilisation figure
is drawn from.

The serving rule refuses any admission that would carry a cell-band past `kpi.capacity.max_admission_utilisation`, so
no cell can appear above the ceiling. A cell *at* the ceiling is one that turned traffic away, and that traffic shows
up in the served rate, not here.

In [15]:
display(results["cell_band_utilisation"])
display(results["prb_usage_heatmaps"])
display(results["cell_band_load"])

# 48k rows is a series, not a table; show where it runs hottest.
usage = results["prb_usage_by_time"]
busiest = (
    usage.sort_values("PRB utilisation", ascending=False)
    .drop_duplicates(["Configuration", "Cell", "Band"])
    .head(10)
    .reset_index(drop=True)
)
display(busiest)

<Figure size 1133x594 with 3 Axes>

<Figure size 1232x572 with 3 Axes>

,Cell,Band,Served reports,Mean PRB load,Peak PRB load,PRB limit,Median SINR [dB],Peak PRB utilisation,Configuration
0,n0c0,2600 MHz,123,12.6004,172.1595,216.0,3.9552,0.7970,Current configuration
1,n0c1,2600 MHz,389,31.6248,170.0246,216.0,8.1041,0.7872,Current configuration
2,n0c2,2600 MHz,103,13.0561,172.7687,216.0,2.1295,0.7999,Current configuration
3,n1c0,2600 MHz,276,37.9955,171.7453,216.0,1.2829,0.7951,Current configuration
4,n1c1,2600 MHz,538,63.5284,172.7813,216.0,2.7273,0.7999,Current configuration
...,...,...,...,...,...,...,...,...,...
31,n2c1,700 MHz,51,2.1796,40.7855,52.0,11.1996,0.7843,TuRBO
32,n2c2,700 MHz,30,1.1372,40.3087,52.0,12.8495,0.7752,TuRBO
33,n3c0,700 MHz,68,1.7912,41.1451,52.0,20.4600,0.7913,TuRBO
34,n3c1,700 MHz,38,1.5395,41.1472,52.0,12.0126,0.7913,TuRBO


,Configuration,Cell,Band,Interval,PRB load,PRB utilisation
0,Current configuration,n1c1,2600 MHz,295,172.7813,0.7999
1,Current configuration,n0c2,2600 MHz,442,172.7687,0.7999
2,TuRBO,n0c1,2600 MHz,148,172.7420,0.7997
3,Current configuration,n2c2,2600 MHz,204,172.7200,0.7996
4,Current configuration,n2c0,1800 MHz,131,84.7339,0.7994
5,Current configuration,n0c0,700 MHz,409,41.5521,0.7991
6,TuRBO,n2c2,2600 MHz,54,172.5854,0.7990
7,TuRBO,n2c1,1800 MHz,227,84.6536,0.7986
8,Current configuration,n1c1,1800 MHz,256,84.6485,0.7986
9,Current configuration,n1c0,1800 MHz,497,84.6422,0.7985


## 13. Tilt Configuration Analysis

$$
\Delta\theta_{i,b} = \theta^{*}_{i,b} - \theta^{(0)}_{i,b}
$$

for the recommended configuration. Negative is uptilt. The cell impact table is sorted by how much traffic each
cell-band gained or lost, so the cells to watch after rollout come first.

In [16]:
display(results["recommended_tilt"])
display(results["tilt_movement_summary"])
display(results["tilt_delta_heatmap"])
display(results["tilt_movement"])
display(results["cell_impact"])

,Cell,Band,Current tilt [°],Proposed tilt [°],Tilt change [°],Minimum tilt [°],Maximum tilt [°]
0,n0c0,2600 MHz,12.0,5.1657,-6.8343,0.0,15.0
1,n0c0,1800 MHz,10.0,0.7395,-9.2605,0.0,15.0
2,n0c0,700 MHz,8.0,5.7165,-2.2835,0.0,15.0
3,n0c1,2600 MHz,12.0,5.5178,-6.4822,0.0,15.0
4,n0c1,1800 MHz,10.0,1.8481,-8.1519,0.0,15.0
5,n0c1,700 MHz,8.0,2.5140,-5.4860,0.0,15.0
6,n0c2,2600 MHz,12.0,14.9504,2.9504,0.0,15.0
7,n0c2,1800 MHz,10.0,0.8368,-9.1632,0.0,15.0
8,n0c2,700 MHz,8.0,3.2318,-4.7682,0.0,15.0
9,n1c0,2600 MHz,12.0,14.5183,2.5183,0.0,15.0


,Band,Cells,Cells moved,Mean absolute tilt change [°],Largest tilt change [°],Mean tilt change [°]
0,1800 MHz,12,12,6.0434,9.2605,-6.0434
1,2600 MHz,12,12,4.8959,9.6307,-3.2095
2,700 MHz,12,12,4.6167,6.9197,-4.6167


<Figure size 803x627 with 2 Axes>

<Figure size 1430x550 with 2 Axes>

,Node,Cell,Azimuth [°],Band,Current tilt [°],Proposed tilt [°],Tilt change [°],"Served reports, before","Peak PRB utilisation, before",Median SINR before [dB],"Served reports, after","Peak PRB utilisation, after",Median SINR after [dB],"Served reports, change","Peak PRB utilisation, change",Median SINR change [dB]
0,n1,n1c1,165.0,1800 MHz,10.0,1.6952,-8.3048,258,0.7986,8.2267,812,0.7953,16.2242,554,-0.0033,7.9974
1,n1,n1c1,165.0,2600 MHz,12.0,14.7459,2.7459,538,0.7999,2.7273,43,0.7897,-2.0227,-495,-0.0102,-4.7501
2,n3,n3c1,165.0,2600 MHz,12.0,3.5518,-8.4482,199,0.7975,0.7477,506,0.7983,2.7622,307,0.0008,2.0145
3,n2,n2c1,165.0,2600 MHz,12.0,13.3611,1.3611,562,0.7946,2.1302,280,0.7961,0.3883,-282,0.0015,-1.7418
4,n3,n3c2,285.0,2600 MHz,12.0,7.1837,-4.8163,652,0.7944,4.9970,871,0.7982,8.0002,219,0.0038,3.0032
5,n1,n1c0,45.0,2600 MHz,12.0,14.5183,2.5183,276,0.7951,1.2829,107,0.7983,-0.8912,-169,0.0032,-2.1741
6,n2,n2c1,165.0,1800 MHz,10.0,4.7785,-5.2215,165,0.7873,4.8821,296,0.7986,7.1272,131,0.0113,2.2451
7,n0,n0c1,165.0,2600 MHz,12.0,5.5178,-6.4822,389,0.7872,8.1041,491,0.7997,10.2591,102,0.0126,2.1551
8,n2,n2c2,285.0,2600 MHz,12.0,3.7887,-8.2113,208,0.7996,4.9409,304,0.7990,7.8460,96,-0.0006,2.9051
9,n1,n1c0,45.0,1800 MHz,10.0,5.1824,-4.8176,68,0.7985,3.8953,163,0.7972,5.5067,95,-0.0013,1.6114


**Observations.** Every one of the 36 cell-bands moved. 31 are uptilted and 5 downtilted, and **all five downtilts are on 2600 MHz**. Mean absolute changes are 6.04° on 1800 MHz, 4.90° on 2600 MHz and 4.62° on 700 MHz, with the largest single change 9.63°. Proposed tilts span 0.74° to 14.95°, so **both** bounds of the [0°, 15°] box are approached - unlike the rule sweep, which pins 1800 and 700 MHz to 0.00° and so presses only the bottom.

The structure is the result, not the spread. 1800 MHz and 700 MHz are uptilted on all twelve cells without exception; 2600 MHz is the only band the search moves in both directions, pushing three cells to within 0.5° of the 15° bound while uptilting the rest.

The cell-impact table shows the trade cell by cell, and the clearest case is n1c1: its **2600 MHz carrier is downtilted 2.75° and loses 495 served reports**, while its **1800 MHz carrier is uptilted 8.30° and gains 554** - the same sector handing its traffic from the high band down to the mid band. n2c1 does the same on a smaller scale (2600 MHz -282, 1800 MHz +131). The large 2600 MHz gains are elsewhere in the network: n3c1 +307 and n3c2 +219, both uptilted. Those are the cells to watch after a rollout.

## 14. Computational Cost

The expensive part is ray tracing, which is summed over every evaluation, apart from algorithm overhead (the difference
to wall clock: GP fitting and acquisition for TuRBO, I/O for all).

In [17]:
cost = results["method_cost"][["Method", "Seed", "Evaluations", "Ray tracing [min]", "Wall clock [min]"]].copy()
cost["Ray tracing per evaluation [s]"] = cost["Ray tracing [min]"] * 60 / cost["Evaluations"]
cost["Overhead [min]"] = cost["Wall clock [min]"] - cost["Ray tracing [min]"]
display(cost)

,Method,Seed,Evaluations,Ray tracing [min],Wall clock [min],Ray tracing per evaluation [s],Overhead [min]
0,Random search,42,145,7.4861,10.2194,3.0977,2.7333
1,Rule-based sweep,42,28,1.0731,1.8539,2.2994,0.7809
2,TuRBO,42,145,9.0052,19.5926,3.7263,10.5874


**Observations.** Ray tracing takes 2.4 s per evaluation for random search and 2.9 s for TuRBO, on the same 145-evaluation budget; the difference is solver variance, not method. TuRBO's GP fitting and acquisition add about 4 minutes on top of its 6.9 minutes of tracing, for 11.1 minutes in all against random search's 7.3. The rule sweep finished in 1.2 minutes and reached a higher objective than random search - on this problem the cheapest method beats the uninformed one, and only TuRBO beats the cheap one, by 0.43 % of J for five times the evaluations.

## 15. Key Results

The recommended TuRBO winner against the current configuration and the best baseline (highest objective among random
search and the rule-based sweep), every UE. The improvement column is TuRBO relative to the current configuration,
positive is better.

In [18]:
board = results["kpi_scoreboard"]
means = board.pivot(index="KPI", columns="Method", values="Mean")
current = board.drop_duplicates("KPI").set_index("KPI")["Current configuration"]
baselines = [m for m in means.columns if m != "TuRBO"]
best_baseline = means.loc["Objective J", baselines].idxmax()
improvement = results["kpi_relative_improvement"].set_index("Method").loc["TuRBO"]
order = list(dict.fromkeys(board["KPI"]))
key_results = pd.DataFrame(
    {
        "Current configuration": current,
        f"Best baseline ({best_baseline})": means[best_baseline],
        "TuRBO": means["TuRBO"],
        "TuRBO improvement [%]": improvement,
    }
).loc[order]
display(key_results)

,Current configuration,Best baseline (Rule-based sweep),TuRBO,TuRBO improvement [%]
Coverage hole rate,0.1125,0.1044,0.1058,5.9553
Co-band overlap rate,0.3164,0.3247,0.3402,-7.5309
Overlap neighbours per covered tile,0.9625,1.1203,1.0643,-10.5725
Weak coverage rate,0.3069,0.2538,0.2638,14.0407
"Cell-edge RSRP, p05 [dBm]",-108.5644,-106.9844,-107.4161,1.0577
"Median RSRP, p50 [dBm]",-84.1068,-80.8430,-81.4024,3.2154
"Cell-edge SINR, p05 [dB]",-5.8729,-5.0047,-5.1535,12.2481
"Median SINR, p50 [dB]",8.4932,9.8325,9.6435,13.5432
Served UE rate,0.5265,0.5930,0.5787,9.9040
Peak PRB utilisation,0.7999,0.7992,0.7997,0.0227


### Main observations

* **Coverage:** TuRBO improves coverage on every area measure - hole rate -6.0 % relative, weak coverage -14.0 %, cell-edge RSRP +1.1 dB, median RSRP +2.7 dB. The rule sweep improves all four by more.
* **Effective coverage:** the share of tiles served by exactly one cell - the quantity the objective is built from - rises from 66.1 % to 69.6 % under TuRBO, the best of the three. This is where TuRBO's win on J comes from.
* **Overlap:** the band-collapsed co-band overlap rate *rises* under all three methods (0.3164 to 0.3402 under TuRBO). On 2600 MHz, the band the objective scores on 71 % of tiles, it falls under all three (0.2010 to 0.1718 under TuRBO). The two unscored layers drift slightly worse. Pile-ups of three or more neighbours fall from 17.6 % to 16.0 % of covered tiles, the best of the three.
* **Signal quality:** median served SINR rises 3.5 dB under TuRBO (5.1 to 8.6), the largest relative move of any measure, and median PRBs per served UE fall 32 %.
* **Band coordination:** TuRBO is the only method that grows the *mid* band - 1800 MHz from 10.5 % to 17.2 % of served reports, while 2600 MHz slips to 33.1 %. Both baselines do the opposite. No layer covers the area alone (per-band hole rates 14-29 % against 11 % for the union).
* **Load:** every configuration sits at the 0.8 admission ceiling, and every method makes the load *less* even (imbalance +3.5 % to +18.8 %). Nothing in the objective asks otherwise.
* **Optimization efficiency:** the rule sweep wins under about 50 evaluations; TuRBO passes it between 50 and 100 and ends 0.0042 above it at 145. Random search stalls after evaluation 62.
* **Objective versus KPIs:** they disagree. TuRBO wins J; the rule sweep wins 8 of the 11 reported KPIs.
* **Robustness:** unknown, with one seed per method.

## 16. Figures for Presentation

All saved under `reports/figures/04_evaluation/`.

| # | Figure | File | Purpose |
|---|---|---|---|
| 1 | Overall KPI comparison | `kpi_improvement.png` | Direct comparison of every method's improvement |
| 2 | Optimization convergence | `search_progress.png` | Sample efficiency of the search |
| 3 | Hole–overlap trade-off | `tradeoff_hole_rate_vs_overlap_rate.png` | Coverage against overlap, with the Pareto front |
| 4 | Coverage before vs after | `coverage_before_after.png`, `coverage_class_maps.png` | Where coverage changed |
| 5 | Serving band distribution | `serving_band_mix.png` | Whether multi-band coordination behaves as intended |
| 6 | KPIs per frequency layer | `band_kpi_panels.png` | Whether each band plays the role its propagation suits |
| 7 | Cell-band PRB usage over time | `prb_usage_heatmaps.png` | Which cells run hot, and when |
| 8 | Tilt adjustment heatmap | `tilt_delta_heatmap.png` | What the optimizer decided |

## 17. Conclusions

1. **Holes:** yes. All three methods lower the hole rate (the sweep -7.2 % relative, TuRBO -6.0 %, random -2.7 %), closing 820, 684 and 459 hole tiles respectively. Per band, every layer improves.
2. **Co-band overlap:** **no, on the reported measure** - all three raise the band-collapsed rate (the sweep +2.6 %, random +6.5 %, TuRBO +7.5 %). On the one band the objective scores it is the reverse: 2600 MHz falls from 0.2010 to 0.1718 (TuRBO), 0.1901 (sweep) and 0.1788 (random). The collapsed rate rises because the unscored 1800 and 700 MHz layers drift worse and because hundreds of newly covered tiles arrive with a neighbour already.
3. **Overlapping neighbours:** the mean per covered tile rises under every method (0.963 to 1.064 under TuRBO), but TuRBO cuts the worst pile-ups - three or more neighbours fall from 17.6 % to 16.0 % of covered tiles. That is the shape of $\lambda e^{1-\lambda}$: steep between one and three contending cells, nearly flat beyond four.
4. **Frequency layers:** the two baselines move served traffic onto the preferred 2600 MHz layer; **TuRBO moves it the other way**, onto 1800 MHz (10.5 % to 17.2 %), by shrinking a few high-band footprints and opening the mid band. No layer covers the area alone.
5. **Load:** no. Every configuration runs at the 0.8 admission ceiling, and every method makes the load less even. Load is reported, not optimised (ADR 0007).
6. **Consistency across runs:** not measurable, with one seed per method.
7. **Evaluations needed:** the rule sweep reaches its result in 25; TuRBO passes it between 50 and 100 and keeps improving to 145, finding its best at evaluation 144 of 145.
8. **Tilt changes:** large and structured. All 36 cell-bands move, 31 uptilted and 5 downtilted, every downtilt on 2600 MHz, the largest change 9.63°, and proposed tilts spanning 0.74° to 14.95° - so both ends of the [0°, 15°] box are in play. The rule sweep instead pins 1800 and 700 MHz to 0.00°.

**Final summary.** TuRBO achieved a 1.86 % higher objective than the current configuration (0.8080 to 0.8231 on the $[0, 1]$ scale of ADR 0009) while improving 9 of the 12 reported measures, and it found a multi-band strategy neither baseline reached: hand traffic from a few contended 2600 MHz sectors down to 1800 MHz. Its margin over the three-variable rule sweep is thin - 0.8231 against 0.8189, so the sweep captures 72 % of the gain for a fifth of the evaluations and a fifteenth of the wall clock - and **the sweep beats TuRBO on 8 of the 11 reported KPIs**, because J does not read signal strength above the hole threshold and that is where the sweep's advantage lies.

The honest reading is that the objective and the report card disagree on this scenario. On a tight ray-tracing budget, or if the KPI table is what the deployment is judged on, the sweep is the better buy; TuRBO earns its cost where effective coverage and the structure of the multi-band layer split are the binding problems.

None of these figures is comparable with anything recorded before 2026-09-20: the objective was replaced (ADR 0009) and its range changed from unbounded to $[0, 1]$. The runs traced under the previous objectives have been deleted rather than archived.

### Threats to validity

| # | Threat | Evidence | Status |
|---|---|---|---|
| 1 | One scenario: every configuration is tuned and scored on the same city and population, so no result measures generalisation | - | Open: held-out scenarios are not implemented |
| 2 | Winner's curse: the best of many evaluations under one solver seed is biased upward | Section 5, recomputed against recorded | Open: re-tracing under other solver seeds is notebook 03b only |
| 3 | Simple reference: the current configuration uses one tilt per band for every cell | Section 7, winner against candidates | Reported |
| 4 | One search seed per method: no spread, interval or test | Section 6, the scoreboard's blank CI columns | Open |
| 5 | The objective depended on three judgement parameters (tau_R, beta and the per-band alpha) | ADR 0007, "Choosing the parameters" | Closed by ADR 0009: the objective has no parameters of its own, so there is nothing left to tune or to disagree on between runs |
| 5b | The coverage utility was unbounded, so J rewarded signal strength above T_cov without limit | ADR 0008, negative consequences | Closed by ADR 0009: J is in [0, 1] and reads only whether a cell clears T_cov, so strength no longer scores. The new exposure is the reverse - an effectively covered but marginal network scores full marks, and only weak_rate and rsrp_p05_dbm show it |
| 5c | The utility's functional form is in code, not in config, so the comparability check cannot in general detect a change to it | ADR 0009; `src/evaluation/runs.py` | Mitigated: the ADR 0009 change did delete the `kpi.objective` block, which the check does compare, so runs either side of it are refused. Every run traced before it was deleted rather than archived |
| 6 | The demand map is built from the MDT, so ground the network already fails to cover carries almost no count | ADR 0007, negative consequences; the fourth hotspot | Mitigated by ADR 0009: the weight is `w = 1 + r`, so unreported ground still weighs 1 against the busiest tile's 2. Under-reported ground is scored at half weight, not written off |
| 6b | With no kernel, the count is zero on 97 % of the grid | ADR 0007, negative consequences | Closed by ADR 0009: with a floor of 1 in the weight, no band is ever scored on the reported tiles alone, so the optimum no longer turns on the UE sampling seed |
| 7 | Hole and weak rates count area, including area without users; the objective counts demand. They can disagree | Section 10.2, coverage by area and by demand | Reported, by design |
| 8 | The served rate and both load measures rely on placeholder capacity settings | `configs/kpi.yaml` `capacity` | Open |
| 9 | Physical simplifications: full-load SINR, Shannon rate without MCS, no calibration against field measurements | Notebook 01, data quality | Open |
| 10 | The rule-based sweep is not budget-matched; tilt movement is not constrained, and the winner moves all 36 cell-bands | Sections 13, 14 | Reported |
| 11 | The project title names Multi-Agent Reinforcement Learning, which is not implemented | - | Open |

## Appendix — Reproducibility

In [19]:
import platform
from importlib.metadata import version

from omegaconf import OmegaConf

setup = results["experiment_setup"].set_index("Parameter")["Setting"]
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
dirty = bool(subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip())
record = {
    "Scenario": setup["Scenario"],
    "Code commit": f"{commit}{' (uncommitted changes)' if dirty else ''}",
    "Runs": ", ".join(f"{m}/{r}" for m, r in results["method_cost"][["Method", "Run"]].itertuples(index=False)),
    "Search seeds": sorted(results["method_cost"]["Seed"].unique().tolist()),
    "Global seed": cfg.seed,
    "Tilt bounds [°]": setup["Tilt bounds [°]"],
    "Tilt step / maximum change": "none (continuous, unconstrained change)",
    "KPI thresholds": OmegaConf.to_container(cfg.kpi, resolve=True),
    "Python": platform.python_version(),
    "Platform": platform.platform(),
    **{name: version(name) for name in ("numpy", "pandas", "scipy", "matplotlib", "hydra-core")},
}
for key, value in record.items():
    print(f"{key}: {value}")

Scenario: scn_7d938e15f9ac4618
Code commit: fa98864d522ecb24982a75463b6978fba403b8a7 (uncommitted changes)
Runs: Random search/2026-09-20_04-09-52, Rule-based sweep/2026-09-20_04-20-14, TuRBO/2026-09-20_03-50-06
Search seeds: [42]
Global seed: 42
Tilt bounds [°]: 0 to 15
Tilt step / maximum change: none (continuous, unconstrained change)
KPI thresholds: {'hole_dbm': -120.0, 'weak_dbm': -90.0, 'overlap_margin_db': 6.0, 'capacity': {'band_preference': ['b2600', 'b1800', 'b700'], 'rsrp_threshold_dbm': -120.0, 'max_admission_utilisation': 0.8, 'throughput_per_ue_bps': 20000000, 'bands': {'b2600': {'scs_hz': 15000}, 'b1800': {'scs_hz': 15000}, 'b700': {'scs_hz': 15000}}}}
Python: 3.13.14
Platform: Windows-11-10.0.26200-SP0
numpy: 2.5.2
pandas: 2.3.3
scipy: 1.18.1
matplotlib: 3.11.1
hydra-core: 1.3.6
